In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")

In [ ]:
train_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.isnull().sum()

### Data Cleaning
Removing unnecessary columns and handling the missing values

In [ ]:
X= train_data.drop(columns=['PassengerId','Name','Ticket','Cabin','Survived'])
y= train_data['Survived']

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
X.isnull().sum()

In [ ]:
X[X['Age'].isnull()]

In [ ]:
X['Age']= X['Age'].fillna(X['Age'].median())

In [ ]:
X['Age'].isnull().sum()

In [ ]:
X['Embarked'].unique()

In [ ]:
X[X['Embarked'].isnull()]

In [ ]:
X = X.drop(index=[61, 829]).reset_index(drop=True)

In [ ]:
y = y.drop(index=[61, 829]).reset_index(drop=True)

In [ ]:
X.isnull().sum()

In [ ]:
X.info()

In [ ]:
X['Sex'].unique()

In [ ]:
X['Sex'] = np.where(X['Sex']=='male',0,1)

In [ ]:
X['Sex'].unique()

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [ ]:
X['Embarked'] = encoder.fit_transform(X['Embarked'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [ ]:
X_train.shape, y_train.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
X_train[['Age','Fare']] = scaler.fit_transform(X_train[['Age','Fare']])

In [ ]:
X_test[['Age','Fare']] = scaler.transform(X_test[['Age','Fare']])

In [ ]:
from sklearn.linear_model import LogisticRegression
logistic = LogisticRegression()

In [ ]:
logistic.fit(X_train,y_train)

In [ ]:
y_pred = logistic.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report
accuracy = accuracy_score(y_test,y_pred)
print("Accuracy: ",accuracy)
print(classification_report(y_test,y_pred))

### Hyperparameter Tuning with cross validation

In [ ]:
model = LogisticRegression()
penalty = ['l1', 'l2', 'elasticnet']
c_values = [100, 10, 1.0, 0.1, 0.01]
solver = ['liblinear', 'lbfgs', 'newton-cg', 'sag', 'saga']

In [ ]:
params = [
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': c_values},
    {'solver': ['lbfgs', 'newton-cg', 'sag'], 'penalty': ['l2'], 'C': c_values},
    {'solver': ['saga'], 'penalty': ['elasticnet'], 'C': c_values, 'l1_ratio': [0.25, 0.5, 0.75]},
]

In [ ]:
from sklearn.model_selection import StratifiedKFold
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
## GridsearchCv -> find the best parameter that fits for this model 
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=model,
    param_grid=params,
    scoring='accuracy',
    cv= CV
)

In [ ]:
grid.fit(X_train,y_train)

In [ ]:
y_pred = grid.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
acc = accuracy_score(y_test,y_pred)
print("Accuracy: ",acc)
print(classification_report(y_test,y_pred))

#### **Randomized Search CV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
random = RandomizedSearchCV(
        estimator=model,
    param_distributions=params,
    scoring='accuracy',
    cv= 5
)

In [ ]:
random.fit(X_train,y_train)

In [ ]:
random.best_params_

In [ ]:
y_pred = random.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report
acc = accuracy_score(y_test,y_pred)
print("Accuracy: ",acc)
print(classification_report(y_test,y_pred))

## The randomsearc cv model gives 80% accuracy.

In [ ]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [ ]:
test_data.head()

In [ ]:
## Scaling the inputs for testing
test_data[['Age','Fare']] = scaler.transform(test_data[['Age','Fare']])

In [ ]:
test_data.head()

In [ ]:
test_data.info()

In [ ]:
test_data_pred = test_data.drop(columns=['PassengerId','Name','Ticket','Cabin'])

In [ ]:
test_data_pred

In [ ]:
test_data_pred['Sex'] = np.where(test_data_pred['Sex']=='male',0,1)

In [ ]:
test_data_pred['Embarked'] = encoder.transform(test_data_pred['Embarked'])

In [ ]:
test_data_pred['Embarked'].value_counts()

In [ ]:
test_data_pred.head()

In [ ]:
test_data_pred.isnull().sum()

In [ ]:
test_data_pred['Age'] = test_data_pred['Age'].fillna(test_data_pred['Age'].median())
test_data_pred['Fare'] = test_data_pred['Fare'].fillna(test_data_pred['Fare'].median())

In [ ]:
pred = random.predict(test_data_pred)

In [ ]:
pred

In [ ]:
pd.read_csv('/kaggle/input/competitions/titanic/gender_submission.csv')

In [ ]:
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': pred
})

In [ ]:
submission

In [ ]:
submission.to_csv('submission.csv', index=False)